# Pose-cluster PCA -- interactive GMM/HDBSCAN exploration

**Kernel:** `abcfold-ago1-mirs-notebook` (`envs/notebook.yaml`) -- install once:
```
conda env create -f envs/notebook.yaml
conda activate abcfold-ago1-mirs-notebook
python -m ipykernel install --user --name abcfold-ago1-mirs-notebook
```

Interactive re-exploration of `results/<complex>/pose_clusters.csv`'s PCA
embedding (`pc1`/`pc2`), i.e. the same data behind the static
`results/<complex>/pose_clusters_pca.svg`. This notebook does **not**
refit PCA or redo the anchor-Kabsch alignment / partner RMSF trim --
`scripts/pose_cluster_anchor.py` computes and persists `pc1`/`pc2` into
`pose_clusters.csv` (added 2026-08-24 specifically so this notebook could
reuse the pipeline's own embedding rather than duplicate it), so what you
see here always matches that SVG exactly. Re-run
`workflows/postprocessing/Snakefile`'s `pose_cluster` rule (or
`scripts/pose_cluster_anchor.py` directly) first if you've changed
clustering parameters (`config.yaml`'s `cluster:` block) and want fresh
numbers here.

Clustering approach mirrors
`../../drb2_modelling/ABCfold_ifb_drbs_dcl4_ds_rna_complexes (sibling)/scripts/_notebook_setup_functions.py`'s
`plot_pca` (`_fit_gmm_bic_sweep` / `_fit_hdbscan_dbcv_search`), adapted
here to a single pre-computed 2-D embedding instead of refitting PCA from
raw multi-dimensional Ca coordinates each call:

```python
df = load_pose_clusters("ago1_fbw2_mir168")

plot_pca(df, complex_name="ago1_fbw2_mir168")                                    # pipeline's own hierarchical-RMSD clusters (default)
plot_pca(df, complex_name="ago1_fbw2_mir168", color_by="backend", cluster_method=None)  # no re-clustering, colour by backend instead
plot_pca(df, complex_name="ago1_fbw2_mir168", color_by="ranking_score", cluster_method=None)  # continuous colour scale

plot_pca(df, complex_name="ago1_fbw2_mir168", cluster_method="gmm", n_components="auto")   # GMM auto: BIC-knee sweep k=1..12
plot_pca(df, complex_name="ago1_fbw2_mir168", cluster_method="gmm", n_components=3)        # GMM manual: fit GMM-3

plot_pca(df, complex_name="ago1_fbw2_mir168", cluster_method="hdbscan", n_components="auto")  # HDBSCAN auto: Optuna/DBCV search
plot_pca(df, complex_name="ago1_fbw2_mir168", cluster_method="hdbscan", n_components="manual",
         hdbscan_min_cluster_size=15)

# Ablation: does the ensemble still separate into the same pose clusters without a given backend?
plot_pca(df, complex_name="ago1_fbw2_mir168", models={"alphafold3": False})
```

**Every `plot_pca` call that shows a real cluster partition (the pipeline's
own `cluster` column, or a fresh `gmm`/`hdbscan` fit -- not plain
`color_by="backend"/"ptm"/"iptm"/"ranking_score"`) automatically symlinks
that clustering's CIFs to disk**, mirrors
`../../drb2_modelling/ABCfold_ifb_drbs_dcl4_ds_rna_complexes (sibling)`'s `_reannotate`:

```text
results/<complex>/reannotated/<method_tag>/cluster_<k>/*.cif   -- up to max_per_cluster=20 CIFs, randomly subsampled
results/<complex>/reannotated/<method_tag>/assignments.csv     -- every model in every cluster, with a `symlinked` column
```

`method_tag` is `"pipeline"` for the default hierarchical-RMSD clusters,
`"gmm_auto_k<k>"`/`"gmm_k<k>"` for GMM, `"hdbscan_auto"`/`"hdbscan_manual"`
for HDBSCAN -- so re-running with different parameters lands in its own
subdirectory rather than overwriting a previous exploration. Point
ChimeraX/PyMOL at a `cluster_<k>/` directory directly to load every
structure in that cluster at once. Pass `reannotate_clusters=False` to
skip the disk writes during quick iteration.

**Every point is one predicted model** (backend x seed x sample); hovering
shows backend/seed/sample_index/cluster/ptm/iptm/ranking_score/cif_path so
you can trace an interesting point straight back to its structure file
under `results/<complex>/`.

**Caveat worth watching for:** backends don't all contribute the same
number of frames (e.g. right now `ago1_fbw2_mir168` only has
AlphaFold3/OpenFold3/RosettaFold3 -- Boltz-2/Chai-1/Protenix all hit
hardware/architecture limits on this complex's size, see project memory /
session history), which can dominate a GMM or HDBSCAN fit by sheer point
count. `models=` ablation above is the way to check whether a cluster
assignment actually depends on that imbalance.

Unlike `ABCfold_NPF_pipeline`'s equivalent notebook, `discover_predictions()`
in `scripts/abcfold_backends.py` already deduplicates RosettaFold3's
`..._model.cif`/`..._model_fixed.cif` pair upstream (at
`scripts/compress_abcfold_metadata.py` time), so there's no
near-duplicate-frame caveat to repeat here.

**Prerequisite:** `workflows/postprocessing/Snakefile`'s `pose_cluster` rule
must have run for a complex before `load_pose_clusters(complex_name)` can
read its `pose_clusters.csv`.


In [1]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import optuna
from hdbscan.validity import validity_index
from kneed import KneeLocator
from sklearn.cluster import HDBSCAN
from sklearn.mixture import GaussianMixture

optuna.logging.set_verbosity(optuna.logging.WARNING)  # one INFO line per trial is too noisy at n_trials=200

ROOT = Path("..")
RESULTS_ROOT = ROOT / "results"
ABCFOLD_ROOT = RESULTS_ROOT / "abcfold"

CLUSTER_PALETTE = px.colors.qualitative.Set1
BACKEND_SYMBOLS = {
    "alphafold3": "circle", "boltz": "square", "chai1": "diamond",
    "openfold3": "triangle-up", "protenix": "x", "rosettafold3": "cross",
}

HDBSCAN_MIN_SAMPLES_CANDIDATES = [3, 5, 10, 15, 20, 25, 30]
HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]
HDBSCAN_CLUSTER_SELECTION_METHODS = ["eom", "leaf"]
HDBSCAN_METRICS = ["euclidean", "cityblock"]


def load_pose_clusters(complex_name: str) -> pd.DataFrame:
    """results/<complex>/pose_clusters.csv, as written by
    scripts/pose_cluster_anchor.py -- pc1/pc2 are that script's own PCA fit
    on the anchor-Kabsch-aligned, RMSF-trimmed partner-chain feature vector
    (see its module docstring for the full algorithm). This notebook does
    NOT refit PCA or redo the Kabsch alignment -- it explores the exact
    embedding the pipeline already computed and persisted, so what you see
    here always matches results/<complex>/pose_clusters_pca.svg."""
    path = RESULTS_ROOT / complex_name / "pose_clusters.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found -- run workflows/postprocessing/Snakefile's pose_cluster "
            f"rule for {complex_name} first (needs scripts/pose_cluster_anchor.py "
            "2026-08-24 or later -- earlier runs didn't persist pc1/pc2)."
        )
    df = pd.read_csv(path)
    if "pc1" not in df.columns:
        raise ValueError(
            f"{path} has no pc1/pc2 columns -- re-run pose_cluster_anchor.py for "
            f"{complex_name} to regenerate it with the current script version."
        )
    var_path = RESULTS_ROOT / complex_name / "pose_clusters_pca_variance.json"
    variance = json.loads(var_path.read_text()) if var_path.exists() else {}
    print(f"[{complex_name}] {len(df)} models, {df['backend'].value_counts().to_dict()}")
    if variance:
        print(f"[{complex_name}] PC1={variance['pc1_explained_variance']:.1%} "
              f"PC2={variance['pc2_explained_variance']:.1%} "
              f"(sum={variance['pc1_explained_variance'] + variance['pc2_explained_variance']:.1%})")
    return df


# ── GMM auto (BIC-knee sweep) ───────────────────────────────────────────────

def _fit_gmm_bic_sweep(xy, k_min=1, k_max=12, n_init=20, random_state=42):
    """Fit a GaussianMixture for every k in [k_min, k_max] and return the
    one sitting at the knee of the BIC-vs-k curve. KneeLocator's default
    interpolation follows every point exactly, so a single noisy BIC value
    reads as a spurious knee right at the first bump -- fitting a
    polynomial through the curve first smooths that out. Falls back to the
    raw BIC minimum if KneeLocator finds no knee. Same approach as
    ABCfold_NPF_pipeline/scripts/_notebook_setup_functions.py's
    _fit_gmm_bic_sweep."""
    k_max = min(k_max, xy.shape[0] - 1)
    ks = list(range(max(1, k_min), k_max + 1))

    gmms, bic_by_k = {}, {}
    for k in ks:
        gmm = GaussianMixture(n_components=k, covariance_type="full",
                               n_init=n_init, random_state=random_state)
        try:
            gmm.fit(xy)
        except ValueError as e:
            print(f"[gmm-auto] WARNING: k={k} failed ({e}), skipping")
            continue
        gmms[k] = gmm
        bic_by_k[k] = float(gmm.bic(xy))

    if not bic_by_k:
        raise RuntimeError(
            f"GMM auto (BIC sweep) failed for every k in [{ks[0]}, {ks[-1]}] -- "
            "try a narrower auto_k_min/auto_k_max range or n_components=<int> (manual)")
    ks = sorted(bic_by_k)
    best_k = ks[int(np.argmin([bic_by_k[k] for k in ks]))]
    if len(ks) >= 3:
        degree = min(7, max(1, len(ks) - 3))
        try:
            kl = KneeLocator(ks, [bic_by_k[k] for k in ks],
                              curve="convex", direction="decreasing",
                              interp_method="polynomial", polynomial_degree=degree)
            if kl.knee is not None:
                best_k = int(kl.knee)
        except Exception as e:
            print(f"[gmm-auto] WARNING: KneeLocator failed ({e}), falling back to BIC minimum")

    return gmms[best_k].predict(xy), best_k, bic_by_k


def _plot_bic_curve(bic_by_k, best_k, title):
    ks = sorted(bic_by_k)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ks, y=[bic_by_k[k] for k in ks], mode="lines+markers",
                              line=dict(color="#1565C0", width=2), marker=dict(size=6), name="BIC"))
    fig.add_trace(go.Scatter(x=[best_k], y=[bic_by_k[best_k]], mode="markers",
                              marker=dict(size=14, color="#d62728", symbol="star"), name=f"knee k={best_k}"))
    fig.update_layout(title=f"{title}<br>BIC sweep k={ks[0]}-{ks[-1]}, knee k={best_k}",
                       xaxis_title="n_components (k)", yaxis_title="BIC",
                       template="plotly_white", height=340, width=480, showlegend=False)
    fig.show()


# ── HDBSCAN auto (Optuna/DBCV search) ───────────────────────────────────────

def _fit_hdbscan_dbcv_search(xy, min_samples_candidates=HDBSCAN_MIN_SAMPLES_CANDIDATES,
                              min_cluster_size_candidates=HDBSCAN_MIN_CLUSTER_SIZE_CANDIDATES,
                              cluster_selection_methods=HDBSCAN_CLUSTER_SELECTION_METHODS,
                              metrics=HDBSCAN_METRICS, n_trials=200, random_state=42):
    """Optuna/TPE search over (min_samples, min_cluster_size,
    cluster_selection_method, metric) for HDBSCAN, scored by DBCV (Moulavi
    et al. 2014) via hdbscan.validity.validity_index. Combos yielding fewer
    than 2 clusters, or erroring inside DBCV, score -1.0 so they're never
    selected. Same approach as ABCfold_NPF_pipeline/scripts/
    _notebook_setup_functions.py's _fit_hdbscan_dbcv_search."""
    n = xy.shape[0]
    max_min_cluster_size = max(2, n // 5)
    candidate_min_cluster_sizes = [m for m in min_cluster_size_candidates if 2 <= m <= max_min_cluster_size]
    if not candidate_min_cluster_sizes:
        candidate_min_cluster_sizes = [max_min_cluster_size]

    grid_size = (len(min_samples_candidates) * len(candidate_min_cluster_sizes)
                 * len(cluster_selection_methods) * len(metrics))
    n_trials = min(n_trials, grid_size)

    def objective(trial):
        min_samples = trial.suggest_categorical("min_samples", list(min_samples_candidates))
        min_cluster_size = trial.suggest_categorical("min_cluster_size", candidate_min_cluster_sizes)
        cluster_selection_method = trial.suggest_categorical("cluster_selection_method", list(cluster_selection_methods))
        metric = trial.suggest_categorical("metric", list(metrics))
        try:
            labels = HDBSCAN(min_samples=min_samples, min_cluster_size=min_cluster_size,
                              cluster_selection_method=cluster_selection_method,
                              metric=metric, copy=False).fit(xy).labels_
            n_clust = len(set(c for c in labels if c >= 0))
            dbcv = float(validity_index(xy.astype(np.float64), labels, metric=metric)) if n_clust >= 2 else -1.0
        except Exception as e:
            print(f"[hdbscan-auto] combo ms={min_samples} mcs={min_cluster_size} "
                  f"{cluster_selection_method}/{metric} failed: {e}")
            labels, dbcv = None, -1.0
        trial.set_user_attr("labels", None if labels is None else labels.tolist())
        return dbcv

    sampler = optuna.samplers.TPESampler(seed=random_state)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_trial = study.best_trial
    best_labels = best_trial.user_attrs["labels"]
    best = {"min_samples": best_trial.params["min_samples"],
            "min_cluster_size": best_trial.params["min_cluster_size"],
            "cluster_selection_method": best_trial.params["cluster_selection_method"],
            "metric": best_trial.params["metric"], "dbcv": best_trial.value}
    labels = np.array(best_labels) if best_labels is not None else np.full(n, -1)
    return labels, best, study


def _plot_dbcv_search(study, best, title):
    dbcvs = sorted(study.trials_dataframe()["value"].fillna(-1.0).tolist(), reverse=True)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=list(range(len(dbcvs))), y=dbcvs, marker_color="#1565C0", name="DBCV"))
    fig.add_hline(y=best["dbcv"], line_dash="dash", line_color="#d62728",
                  annotation_text=f"best DBCV={best['dbcv']:.3f}")
    fig.update_layout(title=f"{title}<br>HDBSCAN Optuna/TPE search, {len(dbcvs)} trials",
                       xaxis_title="trial (sorted by DBCV)", yaxis_title="DBCV",
                       template="plotly_white", height=340, width=480, showlegend=False)
    fig.show()


# ── Reannotate: symlink each cluster's CIFs to disk ─────────────────────────

def reannotate(df: pd.DataFrame, complex_name: str, method_tag: str,
                label_col: str = "_color_label", max_per_cluster: int = 20,
                sample_seed: int = 42, out_dir: Path = None) -> Path:
    """Symlink each cluster's raw ABCfold CIFs (as referenced by df's
    cif_path, relative to results/abcfold/) into
    results/<complex>/reannotated/<method_tag>/cluster_<k>/, so you can
    point ChimeraX/PyMOL/etc. at a whole cluster directory directly instead
    of cross-referencing pose_clusters.csv by hand. Mirrors
    ../../drb2_modelling/ABCfold_ifb_drbs_dcl4_ds_rna_complexes (sibling)/scripts/
    _notebook_setup_functions.py's _reannotate, simplified: this project's
    cif_path is already a single resolvable path per row (no apo/holo
    source_run split to disambiguate), and
    scripts/abcfold_backends.discover_predictions() already deduplicates
    RosettaFold3's raw/_fixed CIF pair upstream (at
    scripts/compress_abcfold_metadata.py time), so there's no collision
    case to skip here.

    Clusters routinely hold more structures than is useful to load at
    once, so at most max_per_cluster per cluster are randomly subsampled
    (without replacement, sample_seed for reproducibility) and only those
    get symlinked to disk -- assignments.csv still lists every row in the
    cluster (with a 'symlinked' column) so full membership stays available
    even though the on-disk CIF set is capped. Any stale symlinks from a
    previous call are removed first, so re-running with different
    clustering parameters doesn't leave orphaned files behind.

    Called automatically by plot_pca whenever it assigns real cluster
    labels (the pipeline's own 'cluster' column, or a fresh gmm/hdbscan
    fit) -- pass reannotate=False to plot_pca to skip it for quick
    iteration."""
    out_dir = (RESULTS_ROOT / complex_name / "reannotated" / method_tag) if out_dir is None else out_dir
    cluster_ids = sorted(df[label_col].unique(), key=str)

    assign_rows = []
    n_symlinked = 0
    for cid in cluster_ids:
        dir_name = str(cid).replace(" ", "_").replace("/", "_")
        cluster_dir = out_dir / dir_name
        cluster_dir.mkdir(parents=True, exist_ok=True)
        for stale in cluster_dir.iterdir():
            if stale.is_symlink():
                stale.unlink()

        cluster_rows = df[df[label_col] == cid]
        sampled_idx = set(cluster_rows.sample(
            n=min(len(cluster_rows), max_per_cluster), random_state=sample_seed,
        ).index)

        for idx, row in cluster_rows.iterrows():
            cif = ABCFOLD_ROOT / row["cif_path"]
            symlinked = idx in sampled_idx and cif.exists()
            if symlinked:
                dest = cluster_dir / f"{row['backend']}_seed{row['seed']}_sample{row['sample_index']}.cif"
                if not dest.exists():
                    dest.symlink_to(cif.resolve())
                    n_symlinked += 1
            assign_rows.append({
                "backend": row["backend"], "seed": row["seed"], "sample_index": row["sample_index"],
                "cluster": cid, "ptm": row["ptm"], "iptm": row["iptm"],
                "ranking_score": row["ranking_score"], "cif_path": row["cif_path"],
                "symlinked": symlinked,
            })
    pd.DataFrame(assign_rows).to_csv(out_dir / "assignments.csv", index=False)
    print(f"[reannotate] {complex_name}/{method_tag}: {n_symlinked} symlinks "
          f"(max {max_per_cluster}/cluster) of {len(assign_rows)} assignments -> {out_dir}")
    return out_dir


# ── Main plotting entry point ───────────────────────────────────────────────

def plot_pca(df: pd.DataFrame, complex_name: str = "", models: dict = None,
             color_by: str = "cluster", cluster_method: str = None, n_components=None,
             auto_k_min: int = 1, auto_k_max: int = 12,
             hdbscan_min_cluster_size=None, hdbscan_min_samples=None,
             hdbscan_cluster_selection_method: str = "eom", hdbscan_metric: str = "euclidean",
             hdbscan_n_trials: int = 200,
             marker_size: int = 7, opacity: float = 0.75,
             reannotate_clusters: bool = True, max_per_cluster: int = 20):
    """Interactive Plotly scatter of one complex's pc1/pc2 (see
    load_pose_clusters -- this does NOT refit PCA, it plots the pipeline's
    own embedding).

    Parameters
    ----------
    df              output of load_pose_clusters(complex_name).
    models          ablation switch -- dict of backend name -> True/False;
                    only rows for backends mapped to True are plotted/
                    clustered. Defaults to every backend present. E.g.
                    models={"alphafold3": False} to ask whether the
                    remaining backends' ensemble still covers the same pose
                    clusters without AF3.
    color_by        "cluster" (default -- the pipeline's own hierarchical
                    RMSD clustering, ignored if cluster_method is set),
                    "backend", "ptm", "iptm", or "ranking_score".
    cluster_method  None (default -- use the existing 'cluster' column),
                    "gmm", or "hdbscan": re-cluster pc1/pc2 fresh with that
                    method instead, overriding color_by.
    n_components    "gmm": int fits exactly that many components (manual);
                    "auto" (default when cluster_method="gmm") sweeps
                    auto_k_min..auto_k_max and picks the BIC-vs-k knee
                    (see _fit_gmm_bic_sweep), with a BIC diagnostic plot.
                    "hdbscan": "auto" (default when cluster_method=
                    "hdbscan") searches hyperparameters via Optuna/TPE
                    scored by DBCV (see _fit_hdbscan_dbcv_search), with a
                    DBCV diagnostic plot; "manual" fits HDBSCAN directly
                    with the explicit hdbscan_* arguments below
                    (hdbscan_min_cluster_size then required). Points
                    HDBSCAN calls noise (-1) are shown as "noise".
    hdbscan_*       explicit HDBSCAN hyperparameters, only used when
                    cluster_method="hdbscan", n_components="manual".
    hdbscan_n_trials  Optuna trial budget for the "auto" HDBSCAN search.
    reannotate_clusters  when a real cluster partition is shown (pipeline's
                    own 'cluster' column, or a fresh gmm/hdbscan fit --
                    NOT for color_by="backend"/"ptm"/"iptm"/"ranking_score"),
                    symlink each cluster's CIFs into results/<complex>/
                    reannotated/<method_tag>/cluster_<k>/ (see reannotate()).
                    Default True, matching ABCfold_NPF_pipeline's notebooks;
                    set False to skip disk writes during quick iteration.
    max_per_cluster  cap on how many CIFs per cluster get symlinked when
                    reannotate_clusters fires (randomly subsampled).
    """
    if models is not None:
        keep = df["backend"].isin([b for b, use in models.items() if use])
        missing = set(models) - set(df["backend"].unique())
        if missing:
            print(f"[plot_pca] note: models={sorted(missing)} not present in this data, ignored")
        df = df.loc[keep].reset_index(drop=True)
        if df.empty:
            raise ValueError(f"models={models!r} leaves no rows -- enable at least one present backend")

    xy = df[["pc1", "pc2"]].to_numpy()
    title = f"{complex_name}: PCA (partner chains)" if complex_name else "PCA (partner chains)"

    if cluster_method == "gmm":
        k_mode = "auto" if n_components in (None, "auto") else n_components
        if k_mode == "auto":
            labels, best_k, bic_by_k = _fit_gmm_bic_sweep(xy, k_min=auto_k_min, k_max=auto_k_max)
            _plot_bic_curve(bic_by_k, best_k, title)
        else:
            gmm = GaussianMixture(n_components=int(k_mode), covariance_type="full", n_init=20, random_state=42)
            labels = gmm.fit_predict(xy)
        df = df.assign(_color_label=[f"gmm {label}" for label in labels])
        color_col, is_categorical = "_color_label", True
        method_tag = f"gmm_auto_k{best_k}" if k_mode == "auto" else f"gmm_k{k_mode}"
        if reannotate_clusters:
            reannotate(df, complex_name, method_tag, max_per_cluster=max_per_cluster)

    elif cluster_method == "hdbscan":
        mode = "auto" if n_components in (None, "auto") else n_components
        if mode == "auto":
            labels, best, study = _fit_hdbscan_dbcv_search(xy, n_trials=hdbscan_n_trials)
            print(f"[hdbscan-auto] best: {best}")
            _plot_dbcv_search(study, best, title)
        else:
            if hdbscan_min_cluster_size is None:
                raise ValueError('cluster_method="hdbscan", n_components="manual" needs hdbscan_min_cluster_size')
            labels = HDBSCAN(min_samples=hdbscan_min_samples, min_cluster_size=hdbscan_min_cluster_size,
                              cluster_selection_method=hdbscan_cluster_selection_method,
                              metric=hdbscan_metric).fit(xy).labels_
        df = df.assign(_color_label=["noise" if label < 0 else f"hdbscan {label}" for label in labels])
        color_col, is_categorical = "_color_label", True
        method_tag = "hdbscan_auto" if mode == "auto" else "hdbscan_manual"
        if reannotate_clusters:
            reannotate(df, complex_name, method_tag, max_per_cluster=max_per_cluster)

    elif color_by == "cluster":
        df = df.assign(_color_label=[f"cluster {c}" for c in df["cluster"]])
        color_col, is_categorical = "_color_label", True
        if reannotate_clusters:
            reannotate(df, complex_name, "pipeline", max_per_cluster=max_per_cluster)
    elif color_by == "backend":
        color_col, is_categorical = "backend", True
    elif color_by in ("ptm", "iptm", "ranking_score"):
        color_col, is_categorical = color_by, False
    else:
        raise ValueError(f"color_by={color_by!r} not recognized (cluster/backend/ptm/iptm/ranking_score)")

    hover_cols = ["backend", "seed", "sample_index", "cluster", "ptm", "iptm", "ranking_score", "cif_path"]
    fig = go.Figure()
    if is_categorical:
        categories = sorted(df[color_col].unique(), key=str)
        palette = CLUSTER_PALETTE if len(categories) <= len(CLUSTER_PALETTE) else px.colors.qualitative.Alphabet
        for i, cat in enumerate(categories):
            sub = df[df[color_col] == cat]
            for backend in sorted(sub["backend"].unique()):
                bsub = sub[sub["backend"] == backend]
                fig.add_trace(go.Scatter(
                    x=bsub["pc1"], y=bsub["pc2"], mode="markers",
                    marker=dict(size=marker_size, opacity=opacity, color=palette[i % len(palette)],
                                symbol=BACKEND_SYMBOLS.get(backend, "circle"),
                                line=dict(width=0.5, color="white")),
                    name=f"{cat} / {backend}",
                    customdata=bsub[hover_cols],
                    hovertemplate="<br>".join(f"{c}: %{{customdata[{i}]}}" for i, c in enumerate(hover_cols)) + "<extra></extra>",
                ))
    else:
        fig.add_trace(go.Scatter(
            x=df["pc1"], y=df["pc2"], mode="markers",
            marker=dict(size=marker_size, opacity=opacity, color=df[color_col],
                        colorscale="Viridis", showscale=True, colorbar=dict(title=color_by),
                        line=dict(width=0.5, color="white")),
            customdata=df[hover_cols],
            hovertemplate="<br>".join(f"{c}: %{{customdata[{i}]}}" for i, c in enumerate(hover_cols)) + "<extra></extra>",
        ))

    var_path = (RESULTS_ROOT / complex_name / "pose_clusters_pca_variance.json") if complex_name else None
    pca_var = json.loads(var_path.read_text()) if var_path and var_path.exists() else {}
    x_title = (f"PC1 ({pca_var['pc1_explained_variance']:.1%} explained variance)"
               if pca_var else "PC1")
    y_title = (f"PC2 ({pca_var['pc2_explained_variance']:.1%} explained variance)"
               if pca_var else "PC2")

    fig.update_layout(
        title=title, xaxis_title=x_title, yaxis_title=y_title,
        template="plotly_white", height=620, width=820,
        legend=dict(font=dict(size=9)),
    )
    fig.show()
    return df


/opt/homebrew/Cellar/micromamba/2.8.1/envs/abcfold-drbs-notebook/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data


In [2]:
COMPLEXES = [
    "ago1_fbw2",
    "ago1_fbw2_mir165a",
    "ago1_fbw2_mir168",
    "ago1_fbw2_mir393a",
    "ago1_fbw2_ask1_cul1",
    "ago1_fbw2_ask1_cul1_mir168",
]

# results/<complex>/pose_clusters.csv exists for all complexes regardless of
# postprocessing progress (compress -> pose_cluster -> select runs early).
# load_pose_clusters prints an n-models + PC1/PC2 explained-variance line each.
dfs = {name: load_pose_clusters(name) for name in COMPLEXES}


[ago1_fbw2] 500 models, {'alphafold3': 100, 'chai1': 100, 'openfold3': 100, 'protenix': 100, 'rosettafold3': 100}
[ago1_fbw2] PC1=49.8% PC2=23.6% (sum=73.4%)
[ago1_fbw2_mir165a] 500 models, {'alphafold3': 100, 'chai1': 100, 'openfold3': 100, 'protenix': 100, 'rosettafold3': 100}
[ago1_fbw2_mir165a] PC1=44.6% PC2=22.2% (sum=66.7%)
[ago1_fbw2_mir168] 500 models, {'alphafold3': 100, 'chai1': 100, 'openfold3': 100, 'protenix': 100, 'rosettafold3': 100}
[ago1_fbw2_mir168] PC1=45.4% PC2=22.5% (sum=67.9%)
[ago1_fbw2_mir393a] 500 models, {'alphafold3': 100, 'chai1': 100, 'openfold3': 100, 'protenix': 100, 'rosettafold3': 100}
[ago1_fbw2_mir393a] PC1=42.5% PC2=21.1% (sum=63.5%)
[ago1_fbw2_ask1_cul1] 400 models, {'alphafold3': 100, 'openfold3': 100, 'protenix': 100, 'rosettafold3': 100}
[ago1_fbw2_ask1_cul1] PC1=45.4% PC2=21.8% (sum=67.2%)
[ago1_fbw2_ask1_cul1_mir168] 400 models, {'alphafold3': 100, 'openfold3': 100, 'protenix': 100, 'rosettafold3': 100}
[ago1_fbw2_ask1_cul1_mir168] PC1=27.0% PC

## Examples

One `plot_pca` call per cell: each opens its own figure, so you can
zoom/pan/hover one plot without a later cell's output replacing it. Every
section is standalone -- it rebinds its own `df_<complex>` from `dfs`.


### `ago1_fbw2`


In [3]:
df_ago1_fbw2 = dfs["ago1_fbw2"]
df_ago1_fbw2.head()


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.60,0.20,0.37,3,0.000001,-209.734774,-313.841079
1,alphafold3,10,1.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.61,0.19,0.37,3,0.768318,-94.114578,-298.548044
2,alphafold3,10,2.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.61,0.22,0.39,2,0.655278,-217.128883,237.840980
3,alphafold3,10,3.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.61,0.19,0.36,3,1.075975,-61.497400,-264.473714
4,alphafold3,10,4.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.61,0.22,0.39,2,0.899075,-213.580182,238.683460


In [4]:
plot_pca(df_ago1_fbw2, complex_name="ago1_fbw2")  # pipeline's own hierarchical-RMSD clusters


[reannotate] ago1_fbw2/pipeline: 57 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.600000,0.200000,0.370000,3,0.000001,-209.734774,-313.841079,cluster 3
1,alphafold3,10,1.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.370000,3,0.768318,-94.114578,-298.548044,cluster 3
2,alphafold3,10,2.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.655278,-217.128883,237.840980,cluster 2
3,alphafold3,10,3.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.360000,3,1.075975,-61.497400,-264.473714,cluster 3
4,alphafold3,10,4.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.899075,-213.580182,238.683460,cluster 2
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.609012,0.275157,-99.658096,2,1.918504,-383.870179,127.538720,cluster 2
496,rosettafold3,9,1.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606292,0.272890,-99.660400,3,2.681404,-137.032312,-260.524292,cluster 3
497,rosettafold3,9,2.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.607992,0.275017,-99.658401,2,2.742952,-288.529170,210.168443,cluster 2
498,rosettafold3,9,3.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606901,0.273243,-99.660004,2,2.728887,-261.259105,215.474566,cluster 2


In [5]:
plot_pca(df_ago1_fbw2, complex_name="ago1_fbw2", color_by="backend", cluster_method=None)  # coloured by backend


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.600000,0.200000,0.370000,3,0.000001,-209.734774,-313.841079
1,alphafold3,10,1.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.370000,3,0.768318,-94.114578,-298.548044
2,alphafold3,10,2.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.655278,-217.128883,237.840980
3,alphafold3,10,3.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.360000,3,1.075975,-61.497400,-264.473714
4,alphafold3,10,4.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.899075,-213.580182,238.683460
...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.609012,0.275157,-99.658096,2,1.918504,-383.870179,127.538720
496,rosettafold3,9,1.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606292,0.272890,-99.660400,3,2.681404,-137.032312,-260.524292
497,rosettafold3,9,2.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.607992,0.275017,-99.658401,2,2.742952,-288.529170,210.168443
498,rosettafold3,9,3.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606901,0.273243,-99.660004,2,2.728887,-261.259105,215.474566


In [6]:
plot_pca(df_ago1_fbw2, complex_name="ago1_fbw2", cluster_method="gmm", n_components="auto")  # GMM auto (BIC-knee)


[reannotate] ago1_fbw2/gmm_auto_k4: 76 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2/reannotated/gmm_auto_k4


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.600000,0.200000,0.370000,3,0.000001,-209.734774,-313.841079,gmm 2
1,alphafold3,10,1.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.370000,3,0.768318,-94.114578,-298.548044,gmm 2
2,alphafold3,10,2.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.655278,-217.128883,237.840980,gmm 3
3,alphafold3,10,3.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.360000,3,1.075975,-61.497400,-264.473714,gmm 2
4,alphafold3,10,4.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.899075,-213.580182,238.683460,gmm 3
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.609012,0.275157,-99.658096,2,1.918504,-383.870179,127.538720,gmm 0
496,rosettafold3,9,1.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606292,0.272890,-99.660400,3,2.681404,-137.032312,-260.524292,gmm 2
497,rosettafold3,9,2.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.607992,0.275017,-99.658401,2,2.742952,-288.529170,210.168443,gmm 3
498,rosettafold3,9,3.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606901,0.273243,-99.660004,2,2.728887,-261.259105,215.474566,gmm 3


In [7]:
plot_pca(df_ago1_fbw2, complex_name="ago1_fbw2", cluster_method="gmm", n_components=4)  # GMM manual k -- adjust per complex


[reannotate] ago1_fbw2/gmm_k4: 76 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2/reannotated/gmm_k4


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.600000,0.200000,0.370000,3,0.000001,-209.734774,-313.841079,gmm 2
1,alphafold3,10,1.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.370000,3,0.768318,-94.114578,-298.548044,gmm 2
2,alphafold3,10,2.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.655278,-217.128883,237.840980,gmm 3
3,alphafold3,10,3.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.360000,3,1.075975,-61.497400,-264.473714,gmm 2
4,alphafold3,10,4.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.899075,-213.580182,238.683460,gmm 3
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.609012,0.275157,-99.658096,2,1.918504,-383.870179,127.538720,gmm 0
496,rosettafold3,9,1.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606292,0.272890,-99.660400,3,2.681404,-137.032312,-260.524292,gmm 2
497,rosettafold3,9,2.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.607992,0.275017,-99.658401,2,2.742952,-288.529170,210.168443,gmm 3
498,rosettafold3,9,3.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606901,0.273243,-99.660004,2,2.728887,-261.259105,215.474566,gmm 3


In [8]:
plot_pca(df_ago1_fbw2, complex_name="ago1_fbw2", cluster_method="hdbscan", n_components="auto")  # HDBSCAN auto (Optuna/DBCV)


[hdbscan-auto] best: {'min_samples': 30, 'min_cluster_size': 20, 'cluster_selection_method': 'eom', 'metric': 'euclidean', 'dbcv': 0.6469039670208154}


[reannotate] ago1_fbw2/hdbscan_auto: 75 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2/reannotated/hdbscan_auto


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.600000,0.200000,0.370000,3,0.000001,-209.734774,-313.841079,hdbscan 0
1,alphafold3,10,1.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.370000,3,0.768318,-94.114578,-298.548044,hdbscan 0
2,alphafold3,10,2.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.655278,-217.128883,237.840980,hdbscan 1
3,alphafold3,10,3.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.190000,0.360000,3,1.075975,-61.497400,-264.473714,hdbscan 0
4,alphafold3,10,4.0,ago1_fbw2/alphafold3_ago1_fbw2/seed-10_sample-...,0.610000,0.220000,0.390000,2,0.899075,-213.580182,238.683460,hdbscan 1
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.609012,0.275157,-99.658096,2,1.918504,-383.870179,127.538720,hdbscan 1
496,rosettafold3,9,1.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606292,0.272890,-99.660400,3,2.681404,-137.032312,-260.524292,hdbscan 0
497,rosettafold3,9,2.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.607992,0.275017,-99.658401,2,2.742952,-288.529170,210.168443,hdbscan 1
498,rosettafold3,9,3.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606901,0.273243,-99.660004,2,2.728887,-261.259105,215.474566,hdbscan 1


In [9]:
plot_pca(df_ago1_fbw2, complex_name="ago1_fbw2", models={**{b: True for b in df_ago1_fbw2["backend"].unique()}, "alphafold3": False})  # ablation: AF3 excluded


[reannotate] ago1_fbw2/pipeline: 57 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,chai1,1,0.0,ago1_fbw2/chai1_ago1_fbw2/chai_output_seed-1/p...,0.741421,0.574504,0.607887,1,2.167498,284.941009,-25.218320,cluster 1
1,chai1,1,1.0,ago1_fbw2/chai1_ago1_fbw2/chai_output_seed-1/p...,0.741158,0.569626,0.603933,1,1.357984,282.033307,-22.206871,cluster 1
2,chai1,1,2.0,ago1_fbw2/chai1_ago1_fbw2/chai_output_seed-1/p...,0.742019,0.575049,0.608443,1,2.186570,283.918783,-13.291227,cluster 1
3,chai1,1,3.0,ago1_fbw2/chai1_ago1_fbw2/chai_output_seed-1/p...,0.740727,0.572372,0.606043,1,1.535130,288.680742,-12.919826,cluster 1
4,chai1,1,4.0,ago1_fbw2/chai1_ago1_fbw2/chai_output_seed-1/p...,0.740584,0.573362,0.606807,1,1.849560,273.407684,-15.951344,cluster 1
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.609012,0.275157,-99.658096,2,1.918504,-383.870179,127.538720,cluster 2
396,rosettafold3,9,1.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606292,0.272890,-99.660400,3,2.681404,-137.032312,-260.524292,cluster 3
397,rosettafold3,9,2.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.607992,0.275017,-99.658401,2,2.742952,-288.529170,210.168443,cluster 2
398,rosettafold3,9,3.0,ago1_fbw2/rosettafold_ago1_fbw2/rosettafold_re...,0.606901,0.273243,-99.660004,2,2.728887,-261.259105,215.474566,cluster 2


### `ago1_fbw2_mir165a`


In [10]:
df_ago1_fbw2_mir165a = dfs["ago1_fbw2_mir165a"]
df_ago1_fbw2_mir165a.head()


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.68,0.62,0.73,2,9.033089e-07,184.369399,172.820558
1,alphafold3,10,1.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.69,0.62,0.74,2,6.807782e-01,206.166231,151.969428
2,alphafold3,10,2.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.68,0.61,0.73,2,4.303755e-01,383.817317,48.986591
3,alphafold3,10,3.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.68,0.61,0.73,2,5.970820e-01,420.815548,61.644233
4,alphafold3,10,4.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.68,0.62,0.74,2,6.508756e-01,425.079412,-4.508256


In [11]:
plot_pca(df_ago1_fbw2_mir165a, complex_name="ago1_fbw2_mir165a")  # pipeline's own hierarchical-RMSD clusters


[reannotate] ago1_fbw2_mir165a/pipeline: 84 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir165a/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.730000,2,9.033089e-07,184.369399,172.820558,cluster 2
1,alphafold3,10,1.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.690000,0.620000,0.740000,2,6.807782e-01,206.166231,151.969428,cluster 2
2,alphafold3,10,2.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,4.303755e-01,383.817317,48.986591,cluster 2
3,alphafold3,10,3.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,5.970820e-01,420.815548,61.644233,cluster 2
4,alphafold3,10,4.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.740000,2,6.508756e-01,425.079412,-4.508256,cluster 2
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.629982,0.530377,-99.449699,5,1.462315e+00,-565.183286,22.983845,cluster 5
496,rosettafold3,9,1.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627363,0.530865,-99.449799,1,1.327854e+00,88.782606,-143.810474,cluster 1
497,rosettafold3,9,2.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.632168,0.531889,-99.448097,5,2.043617e+00,-481.656709,-42.819976,cluster 5
498,rosettafold3,9,3.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627970,0.530381,-99.450104,3,1.581528e+00,-148.202530,91.562883,cluster 3


In [12]:
plot_pca(df_ago1_fbw2_mir165a, complex_name="ago1_fbw2_mir165a", color_by="backend", cluster_method=None)  # coloured by backend


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.730000,2,9.033089e-07,184.369399,172.820558
1,alphafold3,10,1.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.690000,0.620000,0.740000,2,6.807782e-01,206.166231,151.969428
2,alphafold3,10,2.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,4.303755e-01,383.817317,48.986591
3,alphafold3,10,3.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,5.970820e-01,420.815548,61.644233
4,alphafold3,10,4.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.740000,2,6.508756e-01,425.079412,-4.508256
...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.629982,0.530377,-99.449699,5,1.462315e+00,-565.183286,22.983845
496,rosettafold3,9,1.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627363,0.530865,-99.449799,1,1.327854e+00,88.782606,-143.810474
497,rosettafold3,9,2.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.632168,0.531889,-99.448097,5,2.043617e+00,-481.656709,-42.819976
498,rosettafold3,9,3.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627970,0.530381,-99.450104,3,1.581528e+00,-148.202530,91.562883


In [13]:
plot_pca(df_ago1_fbw2_mir165a, complex_name="ago1_fbw2_mir165a", cluster_method="gmm", n_components="auto")  # GMM auto (BIC-knee)


[reannotate] ago1_fbw2_mir165a/gmm_auto_k3: 57 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir165a/reannotated/gmm_auto_k3


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.730000,2,9.033089e-07,184.369399,172.820558,gmm 0
1,alphafold3,10,1.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.690000,0.620000,0.740000,2,6.807782e-01,206.166231,151.969428,gmm 0
2,alphafold3,10,2.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,4.303755e-01,383.817317,48.986591,gmm 0
3,alphafold3,10,3.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,5.970820e-01,420.815548,61.644233,gmm 0
4,alphafold3,10,4.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.740000,2,6.508756e-01,425.079412,-4.508256,gmm 0
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.629982,0.530377,-99.449699,5,1.462315e+00,-565.183286,22.983845,gmm 2
496,rosettafold3,9,1.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627363,0.530865,-99.449799,1,1.327854e+00,88.782606,-143.810474,gmm 1
497,rosettafold3,9,2.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.632168,0.531889,-99.448097,5,2.043617e+00,-481.656709,-42.819976,gmm 2
498,rosettafold3,9,3.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627970,0.530381,-99.450104,3,1.581528e+00,-148.202530,91.562883,gmm 2


In [14]:
plot_pca(df_ago1_fbw2_mir165a, complex_name="ago1_fbw2_mir165a", cluster_method="gmm", n_components=4)  # GMM manual k -- adjust per complex


[reannotate] ago1_fbw2_mir165a/gmm_k4: 74 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir165a/reannotated/gmm_k4


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.730000,2,9.033089e-07,184.369399,172.820558,gmm 0
1,alphafold3,10,1.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.690000,0.620000,0.740000,2,6.807782e-01,206.166231,151.969428,gmm 0
2,alphafold3,10,2.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,4.303755e-01,383.817317,48.986591,gmm 0
3,alphafold3,10,3.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,5.970820e-01,420.815548,61.644233,gmm 0
4,alphafold3,10,4.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.740000,2,6.508756e-01,425.079412,-4.508256,gmm 0
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.629982,0.530377,-99.449699,5,1.462315e+00,-565.183286,22.983845,gmm 1
496,rosettafold3,9,1.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627363,0.530865,-99.449799,1,1.327854e+00,88.782606,-143.810474,gmm 3
497,rosettafold3,9,2.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.632168,0.531889,-99.448097,5,2.043617e+00,-481.656709,-42.819976,gmm 1
498,rosettafold3,9,3.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627970,0.530381,-99.450104,3,1.581528e+00,-148.202530,91.562883,gmm 3


In [15]:
plot_pca(df_ago1_fbw2_mir165a, complex_name="ago1_fbw2_mir165a", cluster_method="hdbscan", n_components="auto")  # HDBSCAN auto (Optuna/DBCV)


[hdbscan-auto] best: {'min_samples': 30, 'min_cluster_size': 20, 'cluster_selection_method': 'eom', 'metric': 'cityblock', 'dbcv': 0.6329784366172453}


[reannotate] ago1_fbw2_mir165a/hdbscan_auto: 72 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir165a/reannotated/hdbscan_auto


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.730000,2,9.033089e-07,184.369399,172.820558,hdbscan 1
1,alphafold3,10,1.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.690000,0.620000,0.740000,2,6.807782e-01,206.166231,151.969428,hdbscan 1
2,alphafold3,10,2.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,4.303755e-01,383.817317,48.986591,hdbscan 1
3,alphafold3,10,3.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.610000,0.730000,2,5.970820e-01,420.815548,61.644233,hdbscan 1
4,alphafold3,10,4.0,ago1_fbw2_mir165a/alphafold3_ago1_fbw2_mir165a...,0.680000,0.620000,0.740000,2,6.508756e-01,425.079412,-4.508256,hdbscan 1
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.629982,0.530377,-99.449699,5,1.462315e+00,-565.183286,22.983845,hdbscan 2
496,rosettafold3,9,1.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627363,0.530865,-99.449799,1,1.327854e+00,88.782606,-143.810474,noise
497,rosettafold3,9,2.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.632168,0.531889,-99.448097,5,2.043617e+00,-481.656709,-42.819976,hdbscan 2
498,rosettafold3,9,3.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627970,0.530381,-99.450104,3,1.581528e+00,-148.202530,91.562883,noise


In [16]:
plot_pca(df_ago1_fbw2_mir165a, complex_name="ago1_fbw2_mir165a", models={**{b: True for b in df_ago1_fbw2_mir165a["backend"].unique()}, "alphafold3": False})  # ablation: AF3 excluded


[reannotate] ago1_fbw2_mir165a/pipeline: 84 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_mir165a/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,chai1,1,0.0,ago1_fbw2_mir165a/chai1_ago1_fbw2_mir165a/chai...,0.756720,0.667178,0.685086,2,1.508511,272.280621,64.071839,cluster 2
1,chai1,1,1.0,ago1_fbw2_mir165a/chai1_ago1_fbw2_mir165a/chai...,0.759151,0.667516,0.685843,2,1.414994,256.323113,41.142090,cluster 2
2,chai1,1,2.0,ago1_fbw2_mir165a/chai1_ago1_fbw2_mir165a/chai...,0.720030,0.634487,0.651596,2,1.567261,131.411317,-90.060250,cluster 2
3,chai1,1,3.0,ago1_fbw2_mir165a/chai1_ago1_fbw2_mir165a/chai...,0.757154,0.670752,0.688032,2,1.285599,301.790535,21.102071,cluster 2
4,chai1,1,4.0,ago1_fbw2_mir165a/chai1_ago1_fbw2_mir165a/chai...,0.758752,0.666626,0.685051,2,1.325704,226.406668,55.999921,cluster 2
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.629982,0.530377,-99.449699,5,1.462315,-565.183286,22.983845,cluster 5
396,rosettafold3,9,1.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627363,0.530865,-99.449799,1,1.327854,88.782606,-143.810474,cluster 1
397,rosettafold3,9,2.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.632168,0.531889,-99.448097,5,2.043617,-481.656709,-42.819976,cluster 5
398,rosettafold3,9,3.0,ago1_fbw2_mir165a/rosettafold_ago1_fbw2_mir165...,0.627970,0.530381,-99.450104,3,1.581528,-148.202530,91.562883,cluster 3


### `ago1_fbw2_mir168`


In [17]:
df_ago1_fbw2_mir168 = dfs["ago1_fbw2_mir168"]
df_ago1_fbw2_mir168.head()


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.70,0.60,0.71,2,0.000001,479.000991,134.666576
1,alphafold3,10,1.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.70,0.62,0.74,2,0.520879,256.100191,230.042922
2,alphafold3,10,2.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.69,0.57,0.70,2,0.321993,477.264234,139.188158
3,alphafold3,10,3.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.69,0.59,0.72,2,0.555535,474.520586,179.070553
4,alphafold3,10,4.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.69,0.58,0.70,2,0.606960,494.637999,105.833881


In [18]:
plot_pca(df_ago1_fbw2_mir168, complex_name="ago1_fbw2_mir168")  # pipeline's own hierarchical-RMSD clusters


[reannotate] ago1_fbw2_mir168/pipeline: 58 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir168/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.600000,0.710000,2,0.000001,479.000991,134.666576,cluster 2
1,alphafold3,10,1.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.620000,0.740000,2,0.520879,256.100191,230.042922,cluster 2
2,alphafold3,10,2.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.570000,0.700000,2,0.321993,477.264234,139.188158,cluster 2
3,alphafold3,10,3.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.590000,0.720000,2,0.555535,474.520586,179.070553,cluster 2
4,alphafold3,10,4.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.580000,0.700000,2,0.606960,494.637999,105.833881,cluster 2
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.631655,0.530112,-99.449600,1,2.169834,-338.878101,242.346194,cluster 1
496,rosettafold3,9,1.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.626815,0.531745,-99.449203,3,1.429397,11.833896,62.343667,cluster 3
497,rosettafold3,9,2.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.629010,0.533585,0.552700,1,1.573613,-398.047872,234.994347,cluster 1
498,rosettafold3,9,3.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.628482,0.531856,-99.448799,1,1.123933,-381.399465,204.945362,cluster 1


In [19]:
plot_pca(df_ago1_fbw2_mir168, complex_name="ago1_fbw2_mir168", color_by="backend", cluster_method=None)  # coloured by backend


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.600000,0.710000,2,0.000001,479.000991,134.666576
1,alphafold3,10,1.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.620000,0.740000,2,0.520879,256.100191,230.042922
2,alphafold3,10,2.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.570000,0.700000,2,0.321993,477.264234,139.188158
3,alphafold3,10,3.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.590000,0.720000,2,0.555535,474.520586,179.070553
4,alphafold3,10,4.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.580000,0.700000,2,0.606960,494.637999,105.833881
...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.631655,0.530112,-99.449600,1,2.169834,-338.878101,242.346194
496,rosettafold3,9,1.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.626815,0.531745,-99.449203,3,1.429397,11.833896,62.343667
497,rosettafold3,9,2.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.629010,0.533585,0.552700,1,1.573613,-398.047872,234.994347
498,rosettafold3,9,3.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.628482,0.531856,-99.448799,1,1.123933,-381.399465,204.945362


In [20]:
plot_pca(df_ago1_fbw2_mir168, complex_name="ago1_fbw2_mir168", cluster_method="gmm", n_components="auto")  # GMM auto (BIC-knee)


[reannotate] ago1_fbw2_mir168/gmm_auto_k6: 117 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir168/reannotated/gmm_auto_k6


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.600000,0.710000,2,0.000001,479.000991,134.666576,gmm 1
1,alphafold3,10,1.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.620000,0.740000,2,0.520879,256.100191,230.042922,gmm 3
2,alphafold3,10,2.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.570000,0.700000,2,0.321993,477.264234,139.188158,gmm 1
3,alphafold3,10,3.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.590000,0.720000,2,0.555535,474.520586,179.070553,gmm 1
4,alphafold3,10,4.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.580000,0.700000,2,0.606960,494.637999,105.833881,gmm 1
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.631655,0.530112,-99.449600,1,2.169834,-338.878101,242.346194,gmm 2
496,rosettafold3,9,1.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.626815,0.531745,-99.449203,3,1.429397,11.833896,62.343667,gmm 3
497,rosettafold3,9,2.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.629010,0.533585,0.552700,1,1.573613,-398.047872,234.994347,gmm 2
498,rosettafold3,9,3.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.628482,0.531856,-99.448799,1,1.123933,-381.399465,204.945362,gmm 2


In [21]:
plot_pca(df_ago1_fbw2_mir168, complex_name="ago1_fbw2_mir168", cluster_method="gmm", n_components=4)  # GMM manual k -- adjust per complex


[reannotate] ago1_fbw2_mir168/gmm_k4: 78 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir168/reannotated/gmm_k4


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.600000,0.710000,2,0.000001,479.000991,134.666576,gmm 0
1,alphafold3,10,1.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.620000,0.740000,2,0.520879,256.100191,230.042922,gmm 0
2,alphafold3,10,2.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.570000,0.700000,2,0.321993,477.264234,139.188158,gmm 0
3,alphafold3,10,3.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.590000,0.720000,2,0.555535,474.520586,179.070553,gmm 0
4,alphafold3,10,4.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.580000,0.700000,2,0.606960,494.637999,105.833881,gmm 0
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.631655,0.530112,-99.449600,1,2.169834,-338.878101,242.346194,gmm 2
496,rosettafold3,9,1.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.626815,0.531745,-99.449203,3,1.429397,11.833896,62.343667,gmm 0
497,rosettafold3,9,2.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.629010,0.533585,0.552700,1,1.573613,-398.047872,234.994347,gmm 2
498,rosettafold3,9,3.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.628482,0.531856,-99.448799,1,1.123933,-381.399465,204.945362,gmm 2


In [22]:
plot_pca(df_ago1_fbw2_mir168, complex_name="ago1_fbw2_mir168", cluster_method="hdbscan", n_components="auto")  # HDBSCAN auto (Optuna/DBCV)


[hdbscan-auto] best: {'min_samples': 10, 'min_cluster_size': 20, 'cluster_selection_method': 'leaf', 'metric': 'euclidean', 'dbcv': 0.41645251425284613}


[reannotate] ago1_fbw2_mir168/hdbscan_auto: 149 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir168/reannotated/hdbscan_auto


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.600000,0.710000,2,0.000001,479.000991,134.666576,hdbscan 0
1,alphafold3,10,1.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.700000,0.620000,0.740000,2,0.520879,256.100191,230.042922,noise
2,alphafold3,10,2.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.570000,0.700000,2,0.321993,477.264234,139.188158,hdbscan 0
3,alphafold3,10,3.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.590000,0.720000,2,0.555535,474.520586,179.070553,hdbscan 0
4,alphafold3,10,4.0,ago1_fbw2_mir168/alphafold3_ago1_fbw2_mir168/s...,0.690000,0.580000,0.700000,2,0.606960,494.637999,105.833881,hdbscan 0
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.631655,0.530112,-99.449600,1,2.169834,-338.878101,242.346194,hdbscan 3
496,rosettafold3,9,1.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.626815,0.531745,-99.449203,3,1.429397,11.833896,62.343667,noise
497,rosettafold3,9,2.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.629010,0.533585,0.552700,1,1.573613,-398.047872,234.994347,noise
498,rosettafold3,9,3.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.628482,0.531856,-99.448799,1,1.123933,-381.399465,204.945362,noise


In [23]:
plot_pca(df_ago1_fbw2_mir168, complex_name="ago1_fbw2_mir168", models={**{b: True for b in df_ago1_fbw2_mir168["backend"].unique()}, "alphafold3": False})  # ablation: AF3 excluded


[reannotate] ago1_fbw2_mir168/pipeline: 56 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_mir168/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,chai1,1,0.0,ago1_fbw2_mir168/chai1_ago1_fbw2_mir168/chai_o...,0.735849,0.621062,0.644019,2,1.274062,295.799771,115.405930,cluster 2
1,chai1,1,1.0,ago1_fbw2_mir168/chai1_ago1_fbw2_mir168/chai_o...,0.736510,0.621589,0.644573,2,1.286569,281.775345,104.773677,cluster 2
2,chai1,1,2.0,ago1_fbw2_mir168/chai1_ago1_fbw2_mir168/chai_o...,0.694693,0.591376,0.612039,3,1.711421,-95.796003,-161.929691,cluster 3
3,chai1,1,3.0,ago1_fbw2_mir168/chai1_ago1_fbw2_mir168/chai_o...,0.735076,0.621241,0.644008,2,1.456152,282.066744,106.144527,cluster 2
4,chai1,1,4.0,ago1_fbw2_mir168/chai1_ago1_fbw2_mir168/chai_o...,0.692845,0.591024,0.611388,1,1.735516,-52.185258,15.655305,cluster 1
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.631655,0.530112,-99.449600,1,2.169834,-338.878101,242.346194,cluster 1
396,rosettafold3,9,1.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.626815,0.531745,-99.449203,3,1.429397,11.833896,62.343667,cluster 3
397,rosettafold3,9,2.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.629010,0.533585,0.552700,1,1.573613,-398.047872,234.994347,cluster 1
398,rosettafold3,9,3.0,ago1_fbw2_mir168/rosettafold_ago1_fbw2_mir168/...,0.628482,0.531856,-99.448799,1,1.123933,-381.399465,204.945362,cluster 1


### `ago1_fbw2_mir393a`


In [24]:
df_ago1_fbw2_mir393a = dfs["ago1_fbw2_mir393a"]
df_ago1_fbw2_mir393a.head()


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.69,0.63,0.74,2,0.000001,147.186953,157.946061
1,alphafold3,10,1.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.69,0.63,0.74,2,0.676640,148.531382,149.673950
2,alphafold3,10,2.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.69,0.63,0.75,2,0.481035,150.095625,205.396196
3,alphafold3,10,3.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.69,0.62,0.74,2,0.538122,381.668758,93.346835
4,alphafold3,10,4.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.68,0.61,0.73,2,0.741643,384.131734,49.950967


In [25]:
plot_pca(df_ago1_fbw2_mir393a, complex_name="ago1_fbw2_mir393a")  # pipeline's own hierarchical-RMSD clusters


[reannotate] ago1_fbw2_mir393a/pipeline: 38 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir393a/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.000001,147.186953,157.946061,cluster 2
1,alphafold3,10,1.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.676640,148.531382,149.673950,cluster 2
2,alphafold3,10,2.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.750000,2,0.481035,150.095625,205.396196,cluster 2
3,alphafold3,10,3.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.620000,0.740000,2,0.538122,381.668758,93.346835,cluster 2
4,alphafold3,10,4.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.680000,0.610000,0.730000,2,0.741643,384.131734,49.950967,cluster 2
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.624771,0.509082,-99.467796,1,2.228121,-623.049928,-220.562347,cluster 1
496,rosettafold3,9,1.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.623927,0.506858,-99.469704,2,1.510445,-153.956214,-289.383198,cluster 2
497,rosettafold3,9,2.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622101,0.505803,-99.470901,2,2.093423,16.343687,-18.943780,cluster 2
498,rosettafold3,9,3.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622960,0.510217,-99.467201,1,1.466292,-506.102754,-159.328225,cluster 1


In [26]:
plot_pca(df_ago1_fbw2_mir393a, complex_name="ago1_fbw2_mir393a", color_by="backend", cluster_method=None)  # coloured by backend


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.000001,147.186953,157.946061
1,alphafold3,10,1.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.676640,148.531382,149.673950
2,alphafold3,10,2.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.750000,2,0.481035,150.095625,205.396196
3,alphafold3,10,3.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.620000,0.740000,2,0.538122,381.668758,93.346835
4,alphafold3,10,4.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.680000,0.610000,0.730000,2,0.741643,384.131734,49.950967
...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.624771,0.509082,-99.467796,1,2.228121,-623.049928,-220.562347
496,rosettafold3,9,1.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.623927,0.506858,-99.469704,2,1.510445,-153.956214,-289.383198
497,rosettafold3,9,2.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622101,0.505803,-99.470901,2,2.093423,16.343687,-18.943780
498,rosettafold3,9,3.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622960,0.510217,-99.467201,1,1.466292,-506.102754,-159.328225


In [27]:
plot_pca(df_ago1_fbw2_mir393a, complex_name="ago1_fbw2_mir393a", cluster_method="gmm", n_components="auto")  # GMM auto (BIC-knee)


[reannotate] ago1_fbw2_mir393a/gmm_auto_k3: 59 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir393a/reannotated/gmm_auto_k3


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.000001,147.186953,157.946061,gmm 0
1,alphafold3,10,1.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.676640,148.531382,149.673950,gmm 0
2,alphafold3,10,2.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.750000,2,0.481035,150.095625,205.396196,gmm 0
3,alphafold3,10,3.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.620000,0.740000,2,0.538122,381.668758,93.346835,gmm 0
4,alphafold3,10,4.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.680000,0.610000,0.730000,2,0.741643,384.131734,49.950967,gmm 0
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.624771,0.509082,-99.467796,1,2.228121,-623.049928,-220.562347,gmm 1
496,rosettafold3,9,1.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.623927,0.506858,-99.469704,2,1.510445,-153.956214,-289.383198,gmm 1
497,rosettafold3,9,2.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622101,0.505803,-99.470901,2,2.093423,16.343687,-18.943780,gmm 1
498,rosettafold3,9,3.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622960,0.510217,-99.467201,1,1.466292,-506.102754,-159.328225,gmm 1


In [28]:
plot_pca(df_ago1_fbw2_mir393a, complex_name="ago1_fbw2_mir393a", cluster_method="gmm", n_components=4)  # GMM manual k -- adjust per complex


[reannotate] ago1_fbw2_mir393a/gmm_k4: 75 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir393a/reannotated/gmm_k4


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.000001,147.186953,157.946061,gmm 1
1,alphafold3,10,1.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.676640,148.531382,149.673950,gmm 1
2,alphafold3,10,2.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.750000,2,0.481035,150.095625,205.396196,gmm 1
3,alphafold3,10,3.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.620000,0.740000,2,0.538122,381.668758,93.346835,gmm 3
4,alphafold3,10,4.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.680000,0.610000,0.730000,2,0.741643,384.131734,49.950967,gmm 3
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.624771,0.509082,-99.467796,1,2.228121,-623.049928,-220.562347,gmm 0
496,rosettafold3,9,1.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.623927,0.506858,-99.469704,2,1.510445,-153.956214,-289.383198,gmm 0
497,rosettafold3,9,2.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622101,0.505803,-99.470901,2,2.093423,16.343687,-18.943780,gmm 0
498,rosettafold3,9,3.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622960,0.510217,-99.467201,1,1.466292,-506.102754,-159.328225,gmm 0


In [29]:
plot_pca(df_ago1_fbw2_mir393a, complex_name="ago1_fbw2_mir393a", cluster_method="hdbscan", n_components="auto")  # HDBSCAN auto (Optuna/DBCV)


[hdbscan-auto] best: {'min_samples': 20, 'min_cluster_size': 40, 'cluster_selection_method': 'eom', 'metric': 'cityblock', 'dbcv': 0.5179826394382457}


[reannotate] ago1_fbw2_mir393a/hdbscan_auto: 72 symlinks (max 20/cluster) of 500 assignments -> ../results/ago1_fbw2_mir393a/reannotated/hdbscan_auto


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.000001,147.186953,157.946061,hdbscan 1
1,alphafold3,10,1.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.740000,2,0.676640,148.531382,149.673950,hdbscan 1
2,alphafold3,10,2.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.630000,0.750000,2,0.481035,150.095625,205.396196,hdbscan 1
3,alphafold3,10,3.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.690000,0.620000,0.740000,2,0.538122,381.668758,93.346835,hdbscan 1
4,alphafold3,10,4.0,ago1_fbw2_mir393a/alphafold3_ago1_fbw2_mir393a...,0.680000,0.610000,0.730000,2,0.741643,384.131734,49.950967,hdbscan 1
...,...,...,...,...,...,...,...,...,...,...,...,...
495,rosettafold3,9,0.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.624771,0.509082,-99.467796,1,2.228121,-623.049928,-220.562347,noise
496,rosettafold3,9,1.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.623927,0.506858,-99.469704,2,1.510445,-153.956214,-289.383198,hdbscan 2
497,rosettafold3,9,2.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622101,0.505803,-99.470901,2,2.093423,16.343687,-18.943780,noise
498,rosettafold3,9,3.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622960,0.510217,-99.467201,1,1.466292,-506.102754,-159.328225,hdbscan 2


In [30]:
plot_pca(df_ago1_fbw2_mir393a, complex_name="ago1_fbw2_mir393a", models={**{b: True for b in df_ago1_fbw2_mir393a["backend"].unique()}, "alphafold3": False})  # ablation: AF3 excluded


[reannotate] ago1_fbw2_mir393a/pipeline: 37 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_mir393a/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,chai1,1,0.0,ago1_fbw2_mir393a/chai1_ago1_fbw2_mir393a/chai...,0.735573,0.647963,0.665485,2,1.319668,223.107321,74.488172,cluster 2
1,chai1,1,1.0,ago1_fbw2_mir393a/chai1_ago1_fbw2_mir393a/chai...,0.736522,0.645898,0.664023,2,1.399123,250.741634,11.810028,cluster 2
2,chai1,1,2.0,ago1_fbw2_mir393a/chai1_ago1_fbw2_mir393a/chai...,0.689144,0.610530,0.626253,1,1.710604,-295.561250,2.063160,cluster 1
3,chai1,1,3.0,ago1_fbw2_mir393a/chai1_ago1_fbw2_mir393a/chai...,0.736061,0.645376,0.663513,2,1.554809,213.430730,60.526292,cluster 2
4,chai1,1,4.0,ago1_fbw2_mir393a/chai1_ago1_fbw2_mir393a/chai...,0.709261,0.626087,0.642722,2,1.849867,87.898488,-95.777083,cluster 2
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.624771,0.509082,-99.467796,1,2.228121,-623.049928,-220.562347,cluster 1
396,rosettafold3,9,1.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.623927,0.506858,-99.469704,2,1.510445,-153.956214,-289.383198,cluster 2
397,rosettafold3,9,2.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622101,0.505803,-99.470901,2,2.093423,16.343687,-18.943780,cluster 2
398,rosettafold3,9,3.0,ago1_fbw2_mir393a/rosettafold_ago1_fbw2_mir393...,0.622960,0.510217,-99.467201,1,1.466292,-506.102754,-159.328225,cluster 1


### `ago1_fbw2_ask1_cul1`


In [31]:
df_ago1_fbw2_ask1_cul1 = dfs["ago1_fbw2_ask1_cul1"]
df_ago1_fbw2_ask1_cul1.head()


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.51,0.45,0.53,2,9.495015e-07,-1003.718640,202.603951
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.51,0.45,0.52,2,6.789176e-01,-1006.379253,165.871777
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.51,0.45,0.53,2,6.375765e-01,-1006.245614,169.902938
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.51,0.44,0.52,2,7.023439e-01,-978.577339,272.328167
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.50,0.44,0.52,2,8.025185e-01,-986.096376,282.085145


In [32]:
plot_pca(df_ago1_fbw2_ask1_cul1, complex_name="ago1_fbw2_ask1_cul1")  # pipeline's own hierarchical-RMSD clusters


[reannotate] ago1_fbw2_ask1_cul1/pipeline: 94 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_ask1_cul1/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,9.495015e-07,-1003.718640,202.603951,cluster 2
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.520000,2,6.789176e-01,-1006.379253,165.871777,cluster 2
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,6.375765e-01,-1006.245614,169.902938,cluster 2
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.440000,0.520000,2,7.023439e-01,-978.577339,272.328167,cluster 2
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.500000,0.440000,0.520000,2,8.025185e-01,-986.096376,282.085145,cluster 2
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452445,0.365052,-99.617500,7,1.897611e+00,512.442580,-226.322523,cluster 7
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452590,0.364758,-99.617699,7,2.159498e+00,480.909574,-237.862251,cluster 7
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.450502,0.364176,-99.618599,7,1.339580e+00,64.574174,-458.003261,cluster 7
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452352,0.364583,-99.617897,7,2.566974e+00,439.378336,-241.123127,cluster 7


In [33]:
plot_pca(df_ago1_fbw2_ask1_cul1, complex_name="ago1_fbw2_ask1_cul1", color_by="backend", cluster_method=None)  # coloured by backend


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,9.495015e-07,-1003.718640,202.603951
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.520000,2,6.789176e-01,-1006.379253,165.871777
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,6.375765e-01,-1006.245614,169.902938
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.440000,0.520000,2,7.023439e-01,-978.577339,272.328167
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.500000,0.440000,0.520000,2,8.025185e-01,-986.096376,282.085145
...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452445,0.365052,-99.617500,7,1.897611e+00,512.442580,-226.322523
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452590,0.364758,-99.617699,7,2.159498e+00,480.909574,-237.862251
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.450502,0.364176,-99.618599,7,1.339580e+00,64.574174,-458.003261
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452352,0.364583,-99.617897,7,2.566974e+00,439.378336,-241.123127


In [34]:
plot_pca(df_ago1_fbw2_ask1_cul1, complex_name="ago1_fbw2_ask1_cul1", cluster_method="gmm", n_components="auto")  # GMM auto (BIC-knee)


[reannotate] ago1_fbw2_ask1_cul1/gmm_auto_k4: 76 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_ask1_cul1/reannotated/gmm_auto_k4


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,9.495015e-07,-1003.718640,202.603951,gmm 2
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.520000,2,6.789176e-01,-1006.379253,165.871777,gmm 2
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,6.375765e-01,-1006.245614,169.902938,gmm 2
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.440000,0.520000,2,7.023439e-01,-978.577339,272.328167,gmm 2
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.500000,0.440000,0.520000,2,8.025185e-01,-986.096376,282.085145,gmm 2
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452445,0.365052,-99.617500,7,1.897611e+00,512.442580,-226.322523,gmm 1
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452590,0.364758,-99.617699,7,2.159498e+00,480.909574,-237.862251,gmm 1
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.450502,0.364176,-99.618599,7,1.339580e+00,64.574174,-458.003261,gmm 0
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452352,0.364583,-99.617897,7,2.566974e+00,439.378336,-241.123127,gmm 1


In [35]:
plot_pca(df_ago1_fbw2_ask1_cul1, complex_name="ago1_fbw2_ask1_cul1", cluster_method="gmm", n_components=4)  # GMM manual k -- adjust per complex


[reannotate] ago1_fbw2_ask1_cul1/gmm_k4: 76 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_ask1_cul1/reannotated/gmm_k4


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,9.495015e-07,-1003.718640,202.603951,gmm 2
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.520000,2,6.789176e-01,-1006.379253,165.871777,gmm 2
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,6.375765e-01,-1006.245614,169.902938,gmm 2
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.440000,0.520000,2,7.023439e-01,-978.577339,272.328167,gmm 2
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.500000,0.440000,0.520000,2,8.025185e-01,-986.096376,282.085145,gmm 2
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452445,0.365052,-99.617500,7,1.897611e+00,512.442580,-226.322523,gmm 1
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452590,0.364758,-99.617699,7,2.159498e+00,480.909574,-237.862251,gmm 1
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.450502,0.364176,-99.618599,7,1.339580e+00,64.574174,-458.003261,gmm 0
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452352,0.364583,-99.617897,7,2.566974e+00,439.378336,-241.123127,gmm 1


In [36]:
plot_pca(df_ago1_fbw2_ask1_cul1, complex_name="ago1_fbw2_ask1_cul1", cluster_method="hdbscan", n_components="auto")  # HDBSCAN auto (Optuna/DBCV)


[hdbscan-auto] best: {'min_samples': 10, 'min_cluster_size': 15, 'cluster_selection_method': 'eom', 'metric': 'euclidean', 'dbcv': 0.5696133485499717}


[reannotate] ago1_fbw2_ask1_cul1/hdbscan_auto: 103 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_ask1_cul1/reannotated/hdbscan_auto


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,9.495015e-07,-1003.718640,202.603951,hdbscan 2
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.520000,2,6.789176e-01,-1006.379253,165.871777,hdbscan 2
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.450000,0.530000,2,6.375765e-01,-1006.245614,169.902938,hdbscan 2
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.510000,0.440000,0.520000,2,7.023439e-01,-978.577339,272.328167,hdbscan 2
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1/alphafold3_ago1_fbw2_ask1_...,0.500000,0.440000,0.520000,2,8.025185e-01,-986.096376,282.085145,hdbscan 2
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452445,0.365052,-99.617500,7,1.897611e+00,512.442580,-226.322523,hdbscan 1
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452590,0.364758,-99.617699,7,2.159498e+00,480.909574,-237.862251,hdbscan 1
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.450502,0.364176,-99.618599,7,1.339580e+00,64.574174,-458.003261,noise
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452352,0.364583,-99.617897,7,2.566974e+00,439.378336,-241.123127,hdbscan 1


In [37]:
plot_pca(df_ago1_fbw2_ask1_cul1, complex_name="ago1_fbw2_ask1_cul1", models={**{b: True for b in df_ago1_fbw2_ask1_cul1["backend"].unique()}, "alphafold3": False})  # ablation: AF3 excluded


[reannotate] ago1_fbw2_ask1_cul1/pipeline: 74 symlinks (max 20/cluster) of 300 assignments -> ../results/ago1_fbw2_ask1_cul1/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,openfold3,1,NaN,ago1_fbw2_ask1_cul1/openfold_ago1_fbw2_ask1_cu...,0.462006,0.396013,0.431287,2,1.122005,-303.496197,102.355788,cluster 2
1,openfold3,1,NaN,ago1_fbw2_ask1_cul1/openfold_ago1_fbw2_ask1_cu...,0.476979,0.416464,0.442033,7,1.559677,597.362908,-159.623400,cluster 7
2,openfold3,1,NaN,ago1_fbw2_ask1_cul1/openfold_ago1_fbw2_ask1_cu...,0.459898,0.395568,0.422121,3,2.016132,-187.997559,-335.184734,cluster 3
3,openfold3,1,NaN,ago1_fbw2_ask1_cul1/openfold_ago1_fbw2_ask1_cu...,0.461827,0.395604,0.428716,6,1.052066,237.718651,1054.473123,cluster 6
4,openfold3,1,NaN,ago1_fbw2_ask1_cul1/openfold_ago1_fbw2_ask1_cu...,0.473075,0.409964,0.432300,7,1.945456,671.055464,-197.560308,cluster 7
...,...,...,...,...,...,...,...,...,...,...,...,...
295,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452445,0.365052,-99.617500,7,1.897611,512.442580,-226.322523,cluster 7
296,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452590,0.364758,-99.617699,7,2.159498,480.909574,-237.862251,cluster 7
297,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.450502,0.364176,-99.618599,7,1.339580,64.574174,-458.003261,cluster 7
298,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1/rosettafold_ago1_fbw2_ask1...,0.452352,0.364583,-99.617897,7,2.566974,439.378336,-241.123127,cluster 7


### `ago1_fbw2_ask1_cul1_mir168`


In [38]:
df_ago1_fbw2_ask1_cul1_mir168 = dfs["ago1_fbw2_ask1_cul1_mir168"]
df_ago1_fbw2_ask1_cul1_mir168.head()


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.48,0.39,0.45,3,0.000001,-923.445570,-226.508565
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.47,0.40,0.47,4,0.651835,265.000923,-286.656583
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.48,0.39,0.47,3,0.524301,-997.570369,-175.135114
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.47,0.39,0.46,7,1.078100,-1090.790951,487.618530
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.47,0.40,0.47,4,1.485742,229.873030,-176.888360


In [39]:
plot_pca(df_ago1_fbw2_ask1_cul1_mir168, complex_name="ago1_fbw2_ask1_cul1_mir168")  # pipeline's own hierarchical-RMSD clusters


[reannotate] ago1_fbw2_ask1_cul1_mir168/pipeline: 121 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_ask1_cul1_mir168/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.450000,3,0.000001,-923.445570,-226.508565,cluster 3
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,0.651835,265.000923,-286.656583,cluster 4
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.470000,3,0.524301,-997.570369,-175.135114,cluster 3
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.390000,0.460000,7,1.078100,-1090.790951,487.618530,cluster 7
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,1.485742,229.873030,-176.888360,cluster 4
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474355,0.398380,-99.586403,6,1.489557,234.999149,-222.333308,cluster 6
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474835,0.398183,-99.586502,4,1.488142,283.790800,-355.820473,cluster 4
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.475760,0.398775,-99.585800,3,1.495118,84.250666,-451.072448,cluster 3
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474068,0.397854,-99.586899,4,1.132498,607.650527,246.073967,cluster 4


In [40]:
plot_pca(df_ago1_fbw2_ask1_cul1_mir168, complex_name="ago1_fbw2_ask1_cul1_mir168", color_by="backend", cluster_method=None)  # coloured by backend


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.450000,3,0.000001,-923.445570,-226.508565
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,0.651835,265.000923,-286.656583
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.470000,3,0.524301,-997.570369,-175.135114
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.390000,0.460000,7,1.078100,-1090.790951,487.618530
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,1.485742,229.873030,-176.888360
...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474355,0.398380,-99.586403,6,1.489557,234.999149,-222.333308
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474835,0.398183,-99.586502,4,1.488142,283.790800,-355.820473
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.475760,0.398775,-99.585800,3,1.495118,84.250666,-451.072448
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474068,0.397854,-99.586899,4,1.132498,607.650527,246.073967


In [41]:
plot_pca(df_ago1_fbw2_ask1_cul1_mir168, complex_name="ago1_fbw2_ask1_cul1_mir168", cluster_method="gmm", n_components="auto")  # GMM auto (BIC-knee)


[reannotate] ago1_fbw2_ask1_cul1_mir168/gmm_auto_k3: 57 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_ask1_cul1_mir168/reannotated/gmm_auto_k3


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.450000,3,0.000001,-923.445570,-226.508565,gmm 0
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,0.651835,265.000923,-286.656583,gmm 1
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.470000,3,0.524301,-997.570369,-175.135114,gmm 0
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.390000,0.460000,7,1.078100,-1090.790951,487.618530,gmm 2
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,1.485742,229.873030,-176.888360,gmm 1
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474355,0.398380,-99.586403,6,1.489557,234.999149,-222.333308,gmm 1
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474835,0.398183,-99.586502,4,1.488142,283.790800,-355.820473,gmm 1
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.475760,0.398775,-99.585800,3,1.495118,84.250666,-451.072448,gmm 0
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474068,0.397854,-99.586899,4,1.132498,607.650527,246.073967,gmm 1


In [42]:
plot_pca(df_ago1_fbw2_ask1_cul1_mir168, complex_name="ago1_fbw2_ask1_cul1_mir168", cluster_method="gmm", n_components=4)  # GMM manual k -- adjust per complex


[reannotate] ago1_fbw2_ask1_cul1_mir168/gmm_k4: 75 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_ask1_cul1_mir168/reannotated/gmm_k4


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.450000,3,0.000001,-923.445570,-226.508565,gmm 2
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,0.651835,265.000923,-286.656583,gmm 1
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.470000,3,0.524301,-997.570369,-175.135114,gmm 2
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.390000,0.460000,7,1.078100,-1090.790951,487.618530,gmm 0
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,1.485742,229.873030,-176.888360,gmm 1
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474355,0.398380,-99.586403,6,1.489557,234.999149,-222.333308,gmm 1
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474835,0.398183,-99.586502,4,1.488142,283.790800,-355.820473,gmm 1
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.475760,0.398775,-99.585800,3,1.495118,84.250666,-451.072448,gmm 2
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474068,0.397854,-99.586899,4,1.132498,607.650527,246.073967,gmm 1


In [43]:
plot_pca(df_ago1_fbw2_ask1_cul1_mir168, complex_name="ago1_fbw2_ask1_cul1_mir168", cluster_method="hdbscan", n_components="auto")  # HDBSCAN auto (Optuna/DBCV)


[hdbscan-auto] best: {'min_samples': 30, 'min_cluster_size': 5, 'cluster_selection_method': 'eom', 'metric': 'euclidean', 'dbcv': 0.329725327154984}


[reannotate] ago1_fbw2_ask1_cul1_mir168/hdbscan_auto: 54 symlinks (max 20/cluster) of 400 assignments -> ../results/ago1_fbw2_ask1_cul1_mir168/reannotated/hdbscan_auto


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,alphafold3,10,0.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.450000,3,0.000001,-923.445570,-226.508565,noise
1,alphafold3,10,1.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,0.651835,265.000923,-286.656583,hdbscan 1
2,alphafold3,10,2.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.480000,0.390000,0.470000,3,0.524301,-997.570369,-175.135114,noise
3,alphafold3,10,3.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.390000,0.460000,7,1.078100,-1090.790951,487.618530,noise
4,alphafold3,10,4.0,ago1_fbw2_ask1_cul1_mir168/alphafold3_ago1_fbw...,0.470000,0.400000,0.470000,4,1.485742,229.873030,-176.888360,hdbscan 1
...,...,...,...,...,...,...,...,...,...,...,...,...
395,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474355,0.398380,-99.586403,6,1.489557,234.999149,-222.333308,hdbscan 1
396,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474835,0.398183,-99.586502,4,1.488142,283.790800,-355.820473,hdbscan 1
397,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.475760,0.398775,-99.585800,3,1.495118,84.250666,-451.072448,hdbscan 1
398,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474068,0.397854,-99.586899,4,1.132498,607.650527,246.073967,hdbscan 1


In [44]:
plot_pca(df_ago1_fbw2_ask1_cul1_mir168, complex_name="ago1_fbw2_ask1_cul1_mir168", models={**{b: True for b in df_ago1_fbw2_ask1_cul1_mir168["backend"].unique()}, "alphafold3": False})  # ablation: AF3 excluded


[reannotate] ago1_fbw2_ask1_cul1_mir168/pipeline: 96 symlinks (max 20/cluster) of 300 assignments -> ../results/ago1_fbw2_ask1_cul1_mir168/reannotated/pipeline


,backend,seed,sample_index,cif_path,ptm,iptm,ranking_score,cluster,core_rmsd_to_ref,pc1,pc2,_color_label
0,openfold3,1,NaN,ago1_fbw2_ask1_cul1_mir168/openfold_ago1_fbw2_...,0.459269,0.389159,0.415322,4,1.588019,318.632312,-263.511140,cluster 4
1,openfold3,1,NaN,ago1_fbw2_ask1_cul1_mir168/openfold_ago1_fbw2_...,0.466686,0.391844,0.416304,4,2.333140,417.815408,-134.707647,cluster 4
2,openfold3,1,NaN,ago1_fbw2_ask1_cul1_mir168/openfold_ago1_fbw2_...,0.460858,0.380142,0.410192,4,1.399951,75.355753,1100.738314,cluster 4
3,openfold3,1,NaN,ago1_fbw2_ask1_cul1_mir168/openfold_ago1_fbw2_...,0.467798,0.395619,0.446479,4,2.182511,309.522366,-190.750554,cluster 4
4,openfold3,1,NaN,ago1_fbw2_ask1_cul1_mir168/openfold_ago1_fbw2_...,0.460527,0.393126,0.421838,4,1.266732,443.419775,26.291224,cluster 4
...,...,...,...,...,...,...,...,...,...,...,...,...
295,rosettafold3,9,0.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474355,0.398380,-99.586403,6,1.489557,234.999149,-222.333308,cluster 6
296,rosettafold3,9,1.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474835,0.398183,-99.586502,4,1.488142,283.790800,-355.820473,cluster 4
297,rosettafold3,9,2.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.475760,0.398775,-99.585800,3,1.495118,84.250666,-451.072448,cluster 3
298,rosettafold3,9,3.0,ago1_fbw2_ask1_cul1_mir168/rosettafold_ago1_fb...,0.474068,0.397854,-99.586899,4,1.132498,607.650527,246.073967,cluster 4
